# SmartTraffic — Final Paper Result Reproduction
## No dataset download · No YOLO extraction · No degradation generation

This notebook is intentionally designed for the **existing Google Drive project** produced by the previous SmartTraffic run.

It will **NOT**:
- download AI City again,
- download UA-DETRAC,
- run YOLO/ByteTrack over video,
- regenerate controlled-degradation video results.

It will:
1. audit everything already present in Google Drive,
2. load clean/degraded `.parquet` caches,
3. reconstruct the exact chronological forecasting split and scalers,
4. load existing model checkpoints (train from cached parquet only if a checkpoint is missing),
5. recompute clean forecasting results,
6. recompute ablation,
7. perform a **paired robustness test with CLEAN future ground truth**,
8. automatically adapt the robustness history length only if the selected degradation videos are too short,
9. run clean and robustness statistical tests,
10. regenerate publication figures and final Excel/CSV/Markdown/ZIP artifacts.

### Important scientific correction
For robustness:

**degraded visual observations → forecasting model → CLEAN future traffic target**

The degraded vehicle count is never used as the future ground truth.

### Output names
To avoid mixing them with the earlier incomplete run, final outputs use the suffix `_FINAL`.

In [ ]:
# ============================================================
# 0. COLAB SETUP — NO DATASET DOWNLOAD
# ============================================================
%pip -q install pyarrow openpyxl statsmodels scipy scikit-learn tqdm tabulate

from google.colab import drive
drive.mount('/content/drive')

import os, sys, math, json, time, random, zipfile, warnings, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("Google Drive mounted.")

In [ ]:
# ============================================================
# 1. CONFIG + PROJECT PATHS
# ============================================================
CFG = {
    "SEED": 42,
    "PROJECT_NAME": "SmartTraffic_VRA_TFormer_IJIES",

    # Must match the completed previous run
    "BIN_SEC": 5,
    "LOOKBACK_STEPS": 60,
    "FORECAST_BLOCK_STEPS": 6,   # 30 sec
    "N_HORIZONS": 4,
    "TRAIN_FRAC": 0.60,
    "VAL_FRAC": 0.20,

    # Model
    "BATCH_SIZE": 64,
    "EPOCHS_FALLBACK": 80,
    "PATIENCE": 12,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "D_MODEL": 64,
    "N_HEAD": 4,
    "N_LAYERS": 2,
    "DROPOUT": 0.10,

    # Robustness:
    # If 60-step history cannot fit the already-created degradation clips,
    # choose the largest history among these that supports >=2 sequences.
    "ROBUST_LOOKBACK_CANDIDATES": [60, 48, 36, 24, 18, 12, 6],
    "MIN_ROBUST_SEQUENCES_DESIRED": 2,

    "FIG_DPI": 400,
}

SEED = CFG["SEED"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path("/content/drive/MyDrive") / CFG["PROJECT_NAME"]

D = {
    "root": ROOT,
    "data": ROOT / "data",
    "cache": ROOT / "cache",
    "clean": ROOT / "cache" / "aicity_clean",
    "degraded": ROOT / "cache" / "aicity_degraded",
    "models": ROOT / "models",
    "tables": ROOT / "tables",
    "figures": ROOT / "figures",
    "logs": ROOT / "logs",
}

for p in [D["models"], D["tables"], D["figures"], D["logs"]]:
    p.mkdir(parents=True, exist_ok=True)

if not ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {ROOT}\n"
        "This resume notebook expects the previous SmartTraffic project in MyDrive."
    )

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

print("Project root:", ROOT)

In [ ]:
# ============================================================
# 2. ASSET AUDIT — WHAT ALREADY EXISTS?
# ============================================================
def files_under(path, pattern="*"):
    path = Path(path)
    if not path.exists():
        return []
    return sorted([p for p in path.rglob(pattern) if p.is_file()])

clean_parts = files_under(D["clean"], "*.parquet")
degraded_parts = files_under(D["degraded"], "*.parquet")
model_parts = files_under(D["models"], "*.pt")
table_parts = files_under(D["tables"])
figure_parts = files_under(D["figures"])
log_parts = files_under(D["logs"])

audit = pd.DataFrame([
    ["Clean sequence parquet", len(clean_parts), str(D["clean"])],
    ["Degraded parquet", len(degraded_parts), str(D["degraded"])],
    ["Model checkpoints (.pt)", len(model_parts), str(D["models"])],
    ["Existing tables/files", len(table_parts), str(D["tables"])],
    ["Existing figures", len(figure_parts), str(D["figures"])],
    ["Existing logs", len(log_parts), str(D["logs"])],
], columns=["Asset", "Count", "Path"])

display(audit)

print("\nExisting checkpoints:")
for p in model_parts:
    print(" -", p.name)

print("\nExisting aggregate caches:")
for p in [
    D["cache"] / "aicity_clean_all.parquet",
    D["cache"] / "aicity_degraded_all.parquet",
]:
    print(" -", p.name, "✅" if p.exists() else "❌")

if not (D["cache"] / "aicity_clean_all.parquet").exists() and len(clean_parts) == 0:
    raise RuntimeError(
        "No CLEAN parquet cache found. Do NOT run further. "
        "The expensive video-extraction cache is missing."
    )

if not (D["cache"] / "aicity_degraded_all.parquet").exists() and len(degraded_parts) == 0:
    raise RuntimeError(
        "No DEGRADED parquet cache found. "
        "The robustness analysis cannot be rebuilt without rerunning degradation."
    )

audit.to_csv(D["tables"] / "resume_asset_audit_FINAL.csv", index=False)

In [ ]:
# ============================================================
# 3. LOAD CLEAN + DEGRADED CACHES
# ============================================================
def load_cache(aggregate_path, parts_root):
    aggregate_path = Path(aggregate_path)
    if aggregate_path.exists():
        print("[cache] loading", aggregate_path.name)
        return pd.read_parquet(aggregate_path)

    parts = sorted(Path(parts_root).rglob("*.parquet"))
    if not parts:
        raise RuntimeError(f"No parquet files found under {parts_root}")

    print("Building aggregate from", len(parts), "parquet files.")
    df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    df.to_parquet(aggregate_path, index=False)
    return df

clean_df = load_cache(
    D["cache"] / "aicity_clean_all.parquet",
    D["clean"]
)

degraded_df = load_cache(
    D["cache"] / "aicity_degraded_all.parquet",
    D["degraded"]
)

clean_df = clean_df.sort_values(["sequence", "bin_idx"]).reset_index(drop=True)
degraded_df = degraded_df.sort_values(
    ["condition", "severity", "sequence", "bin_idx"]
).reset_index(drop=True)

TRAFFIC_COLS = ["car", "motorcycle", "bus", "truck", "total"]
REL_COLS = [
    "mean_conf",
    "track_continuity",
    "blur_log",
    "illumination_quality",
    "occlusion_quality",
    "frame_availability",
]

missing_clean = [c for c in TRAFFIC_COLS + REL_COLS if c not in clean_df.columns]
missing_deg = [c for c in TRAFFIC_COLS + REL_COLS if c not in degraded_df.columns]

if missing_clean:
    raise RuntimeError(f"Clean cache missing columns: {missing_clean}")
if missing_deg:
    raise RuntimeError(f"Degraded cache missing columns: {missing_deg}")

print("Clean rows:", len(clean_df))
print("Clean sequences:", clean_df["sequence"].nunique())
print("Degraded rows:", len(degraded_df))
print("Degraded sequences:", degraded_df["sequence"].nunique())

scenario_inventory = (
    degraded_df
    .groupby(["condition", "severity"])
    .agg(
        sequences=("sequence", "nunique"),
        bins=("bin_idx", "count"),
    )
    .reset_index()
)

display(scenario_inventory)
scenario_inventory.to_csv(
    D["tables"] / "degradation_scenario_inventory_FINAL.csv",
    index=False
)

In [ ]:
# ============================================================
# 4. REBUILD EXACT CLEAN FORECASTING DATA + SCALERS
# ============================================================
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

# Fill reliability descriptors exactly from sequence interpolation,
# then TRAIN-only medians.
for c in REL_COLS:
    clean_df[c] = clean_df.groupby("sequence")[c].transform(
        lambda s: s.interpolate(limit_direction="both")
    )

bounds = {}
for seq, g in clean_df.groupby("sequence"):
    n = len(g)
    tr = int(n * CFG["TRAIN_FRAC"])
    va = int(n * (CFG["TRAIN_FRAC"] + CFG["VAL_FRAC"]))
    bounds[seq] = (tr, va, n)

train_row_mask = np.zeros(len(clean_df), dtype=bool)
for seq, idx in clean_df.groupby("sequence").groups.items():
    tr = bounds[seq][0]
    ids = list(idx)
    train_row_mask[ids[:tr]] = True

rel_median = clean_df.loc[train_row_mask, REL_COLS].median()
for c in REL_COLS:
    clean_df[c] = clean_df[c].fillna(rel_median[c])

x_scaler = StandardScaler().fit(
    clean_df.loc[train_row_mask, TRAFFIC_COLS]
)
q_scaler = StandardScaler().fit(
    clean_df.loc[train_row_mask, REL_COLS]
)

L = CFG["LOOKBACK_STEPS"]
B = CFG["FORECAST_BLOCK_STEPS"]
H = CFG["N_HORIZONS"]

samples = []

for seq, g in clean_df.groupby("sequence", sort=False):
    g = g.reset_index(drop=True)
    tr, va, n = bounds[seq]

    X = x_scaler.transform(g[TRAFFIC_COLS]).astype(np.float32)
    Q = q_scaler.transform(g[REL_COLS]).astype(np.float32)
    y_raw_series = g["total"].values.astype(float)

    future = B * H

    for t in range(L, n - future + 1):
        y_raw = np.array([
            y_raw_series[t + h*B : t + (h+1)*B].sum()
            for h in range(H)
        ], dtype=np.float32)

        target_end = t + future

        if target_end <= tr:
            split = "train"
        elif t >= tr and target_end <= va:
            split = "val"
        elif t >= va:
            split = "test"
        else:
            continue

        samples.append({
            "sequence": seq,
            "split": split,
            "t": int(t),
            "x": X[t-L:t],
            "q": Q[t-L:t],
            "y_raw": y_raw,
        })

train_targets = np.vstack(
    [r["y_raw"] for r in samples if r["split"] == "train"]
)

y_scaler = StandardScaler().fit(
    train_targets.reshape(-1, 1)
)

for r in samples:
    r["y"] = (
        y_scaler
        .transform(r["y_raw"].reshape(-1, 1))
        .reshape(-1)
        .astype(np.float32)
    )

split_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "windows": [
        sum(r["split"] == s for r in samples)
        for s in ["train", "val", "test"]
    ]
})

display(split_summary)

# Expected from the completed earlier run: 798 / 294 / 294.
expected = {"train": 798, "val": 294, "test": 294}
observed = dict(zip(split_summary["split"], split_summary["windows"]))

if observed != expected:
    print(
        "WARNING: reconstructed split differs from the previous run.\n"
        "Previous:", expected, "\nCurrent:", observed
    )
else:
    print("✅ Forecast split exactly matches the previous completed run.")

split_summary.to_csv(
    D["tables"] / "table_forecasting_split_summary_FINAL.csv",
    index=False
)

class WindowDS(Dataset):
    def __init__(self, rows, split):
        self.rows = [r for r in rows if r["split"] == split]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return (
            torch.tensor(r["x"], dtype=torch.float32),
            torch.tensor(r["q"], dtype=torch.float32),
            torch.tensor(r["y"], dtype=torch.float32),
            r["sequence"],
            int(r["t"]),
        )

datasets = {
    s: WindowDS(samples, s)
    for s in ["train", "val", "test"]
}

loaders = {
    "train": DataLoader(
        datasets["train"],
        batch_size=CFG["BATCH_SIZE"],
        shuffle=True,
        num_workers=0
    ),
    "val": DataLoader(
        datasets["val"],
        batch_size=CFG["BATCH_SIZE"],
        shuffle=False,
        num_workers=0
    ),
    "test": DataLoader(
        datasets["test"],
        batch_size=CFG["BATCH_SIZE"],
        shuffle=False,
        num_workers=0
    ),
}

print({k: len(v.dataset) for k, v in loaders.items()})

In [ ]:
# ============================================================
# 5. MODEL DEFINITIONS — EXACT ARCHITECTURES FROM PREVIOUS RUN
# ============================================================
import torch.nn as nn
import torch.nn.functional as F

NT = len(TRAFFIC_COLS)
NR = len(REL_COLS)
NO = CFG["N_HORIZONS"]
DM = CFG["D_MODEL"]

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.r = nn.LSTM(
            NT, DM, 2,
            batch_first=True,
            dropout=CFG["DROPOUT"]
        )
        self.h = nn.Linear(DM, NO)

    def forward(self, x, q=None):
        z, _ = self.r(x)
        return self.h(z[:, -1])

class GRUModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.r = nn.GRU(
            NT, DM, 2,
            batch_first=True,
            dropout=CFG["DROPOUT"]
        )
        self.h = nn.Linear(DM, NO)

    def forward(self, x, q=None):
        z, _ = self.r(x)
        return self.h(z[:, -1])

class TransformerModel(nn.Module):
    def __init__(self, input_dim=NT):
        super().__init__()
        self.p = nn.Linear(input_dim, DM)
        self.pos = PositionalEncoding(DM)

        layer = nn.TransformerEncoderLayer(
            DM,
            CFG["N_HEAD"],
            DM * 4,
            CFG["DROPOUT"],
            batch_first=True,
            norm_first=True
        )
        self.e = nn.TransformerEncoder(
            layer,
            CFG["N_LAYERS"]
        )
        self.h = nn.Sequential(
            nn.LayerNorm(DM),
            nn.Linear(DM, NO)
        )

    def forward(self, x, q=None):
        z = self.e(
            self.pos(self.p(x))
        )
        return self.h(
            z.mean(dim=1)
        )

class ReliabilityConcatModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.m = TransformerModel(
            NT + NR
        )

    def forward(self, x, q=None):
        return self.m(
            torch.cat([x, q], dim=-1)
        )

class PatchTSTModel(nn.Module):
    def __init__(self, patch=12, stride=6):
        super().__init__()
        self.patch = patch
        self.stride = stride

        n_patch = (
            1
            + (CFG["LOOKBACK_STEPS"] - patch)
            // stride
        )

        self.pp = nn.Linear(
            patch,
            DM
        )

        self.pos = nn.Parameter(
            torch.zeros(
                1,
                n_patch,
                DM
            )
        )

        layer = nn.TransformerEncoderLayer(
            DM,
            CFG["N_HEAD"],
            DM * 4,
            CFG["DROPOUT"],
            batch_first=True,
            norm_first=True
        )

        self.e = nn.TransformerEncoder(
            layer,
            CFG["N_LAYERS"]
        )

        self.h = nn.Linear(
            NT * DM,
            NO
        )

    def forward(self, x, q=None):
        p = (
            x.transpose(1, 2)
            .unfold(
                2,
                self.patch,
                self.stride
            )
        )

        B0, C0, P0, PL = p.shape

        z = self.pp(
            p.reshape(
                B0 * C0,
                P0,
                PL
            )
        ) + self.pos[:, :P0]

        z = (
            self.e(z)
            .mean(dim=1)
            .reshape(
                B0,
                C0 * DM
            )
        )

        return self.h(z)

class ITransformerModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.p = nn.Linear(
            CFG["LOOKBACK_STEPS"],
            DM
        )

        self.ve = nn.Parameter(
            torch.zeros(
                1,
                NT,
                DM
            )
        )

        layer = nn.TransformerEncoderLayer(
            DM,
            CFG["N_HEAD"],
            DM * 4,
            CFG["DROPOUT"],
            batch_first=True,
            norm_first=True
        )

        self.e = nn.TransformerEncoder(
            layer,
            CFG["N_LAYERS"]
        )

        self.h = nn.Linear(
            NT * DM,
            NO
        )

    def forward(self, x, q=None):
        z = (
            self.p(
                x.transpose(1, 2)
            )
            + self.ve
        )

        z = (
            self.e(z)
            .reshape(
                x.size(0),
                -1
            )
        )

        return self.h(z)

class ReliabilityAttentionLayer(nn.Module):
    def __init__(self):
        super().__init__()

        self.nh = CFG["N_HEAD"]

        self.att = nn.MultiheadAttention(
            DM,
            self.nh,
            dropout=CFG["DROPOUT"],
            batch_first=True
        )

        self.n1 = nn.LayerNorm(DM)
        self.n2 = nn.LayerNorm(DM)

        self.ff = nn.Sequential(
            nn.Linear(
                DM,
                DM * 4
            ),
            nn.GELU(),
            nn.Dropout(
                CFG["DROPOUT"]
            ),
            nn.Linear(
                DM * 4,
                DM
            ),
        )

    def forward(
        self,
        x,
        reliability=None,
        lam=None
    ):
        mask = None

        if (
            reliability is not None
            and lam is not None
        ):
            B0, L0, _ = reliability.shape

            key_bias = (
                lam
                * torch.log(
                    reliability
                    .squeeze(-1)
                    .clamp_min(1e-5)
                )
            )

            mask = (
                key_bias[:, None, :]
                .expand(
                    B0,
                    L0,
                    L0
                )
                .repeat_interleave(
                    self.nh,
                    dim=0
                )
            )

        h = self.n1(x)

        a, _ = self.att(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + a

        return (
            x
            + self.ff(
                self.n2(x)
            )
        )

class VRATFormer(nn.Module):
    def __init__(
        self,
        use_gate=True,
        use_rel_att=True
    ):
        super().__init__()

        self.use_gate = use_gate
        self.use_rel_att = use_rel_att

        self.obs = nn.Sequential(
            nn.Linear(
                NT,
                DM
            ),
            nn.GELU()
        )

        self.rel = nn.Sequential(
            nn.Linear(
                NR,
                DM // 2
            ),
            nn.GELU()
        )

        self.gate = nn.Sequential(
            nn.Linear(
                DM // 2,
                1
            ),
            nn.Sigmoid()
        )

        self.ctx = nn.GRU(
            NT,
            DM,
            batch_first=True
        )

        self.pos = PositionalEncoding(
            DM
        )

        self.layers = nn.ModuleList([
            ReliabilityAttentionLayer()
            for _ in range(
                CFG["N_LAYERS"]
            )
        ])

        self.rawlam = nn.Parameter(
            torch.tensor(0.0)
        )

        self.head = nn.Sequential(
            nn.LayerNorm(DM),
            nn.Linear(
                DM,
                NO
            )
        )

    def forward(
        self,
        x,
        q=None,
        return_reliability=False
    ):

        obs = self.obs(x)

        ctx, _ = self.ctx(x)

        r = self.gate(
            self.rel(q)
        )

        z = (
            r * obs
            + (1 - r) * ctx
            if self.use_gate
            else obs
        )

        z = self.pos(z)

        lam = (
            F.softplus(
                self.rawlam
            )
            if self.use_rel_att
            else None
        )

        rr = (
            r
            if self.use_rel_att
            else None
        )

        for layer in self.layers:
            z = layer(
                z,
                rr,
                lam
            )

        pred = self.head(
            z.mean(dim=1)
        )

        if return_reliability:
            return pred, r, lam

        return pred

def fresh_models():
    return {
        "LSTM": LSTMModel(),
        "GRU": GRUModel(),
        "Transformer": TransformerModel(),
        "PatchTST": PatchTSTModel(),
        "iTransformer": ITransformerModel(),
        "Transformer+ReliabilityConcat": ReliabilityConcatModel(),
        "VRA-TFormer": VRATFormer(
            True,
            True
        ),
    }

In [ ]:
# ============================================================
# 6. LOAD EXISTING CHECKPOINTS
# FALLBACK: TRAIN FROM PARQUET ONLY IF A CHECKPOINT IS MISSING
# ============================================================
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

def safe_model_name(name):
    return re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        name
    )

def checkpoint_path(name):
    return (
        D["models"]
        / f"{safe_model_name(name)}.pt"
    )

def load_state(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=True
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE
        )

def inverse_target(a):
    a = np.asarray(a)
    shape = a.shape

    return (
        y_scaler
        .inverse_transform(
            a.reshape(-1, 1)
        )
        .reshape(shape)
    )

def run_epoch(
    model,
    loader,
    optimizer=None
):
    training = optimizer is not None

    model.train(training)

    total = 0.0

    for x, q, y, _, _ in loader:
        x = x.to(DEVICE)
        q = q.to(DEVICE)
        y = y.to(DEVICE)

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        pred = model(
            x,
            q
        )

        loss = F.mse_loss(
            pred,
            y
        )

        if training:
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

        total += (
            loss.item()
            * len(x)
        )

    return (
        total
        / max(
            len(loader.dataset),
            1
        )
    )

@torch.no_grad()
def predict_loader(
    model,
    loader
):
    model.eval()

    ys0 = []
    ps0 = []
    seqs = []
    ts = []

    for x, q, y, seq, t in loader:
        p = model(
            x.to(DEVICE),
            q.to(DEVICE)
        ).cpu().numpy()

        ys0.append(
            y.numpy()
        )

        ps0.append(p)

        seqs.extend(
            list(seq)
        )

        ts.extend([
            int(v)
            for v in t
        ])

    y = np.concatenate(
        ys0
    )

    p = np.concatenate(
        ps0
    )

    return (
        inverse_target(y),
        inverse_target(p),
        seqs,
        ts,
    )

def metric_table(
    y_true,
    y_pred,
    model_name
):
    rows = []

    for j in range(NO):

        yt = y_true[:, j]
        yp = y_pred[:, j]

        mae = mean_absolute_error(
            yt,
            yp
        )

        rmse = (
            mean_squared_error(
                yt,
                yp
            ) ** 0.5
        )

        wape = (
            np.abs(
                yt - yp
            ).sum()
            / max(
                np.abs(yt).sum(),
                1e-9
            )
            * 100
        )

        r2 = (
            r2_score(
                yt,
                yp
            )
            if len(
                np.unique(yt)
            ) > 1
            else np.nan
        )

        rows.append({
            "Model": model_name,
            "Horizon": f"{30*(j+1)}s",
            "MAE": mae,
            "RMSE": rmse,
            "WAPE_%": wape,
            "R2": r2,
        })

    return pd.DataFrame(
        rows
    )

def train_fallback(
    name,
    model
):
    print(
        f"[fallback train] {name}"
    )

    model = model.to(
        DEVICE
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["LR"],
        weight_decay=CFG["WEIGHT_DECAY"]
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=0.5,
        patience=4
    )

    best = float("inf")
    wait = 0
    history = []

    for epoch_id in range(
        1,
        CFG["EPOCHS_FALLBACK"] + 1
    ):

        tr_loss = run_epoch(
            model,
            loaders["train"],
            opt
        )

        val_loss = run_epoch(
            model,
            loaders["val"],
            None
        )

        scheduler.step(
            val_loss
        )

        history.append({
            "epoch": epoch_id,
            "train_mse_scaled": tr_loss,
            "val_mse_scaled": val_loss,
        })

        if val_loss < best - 1e-6:
            best = val_loss
            wait = 0

            torch.save(
                model.state_dict(),
                checkpoint_path(name)
            )

        else:
            wait += 1

            if wait >= CFG["PATIENCE"]:
                break

    model.load_state_dict(
        load_state(
            checkpoint_path(name)
        )
    )

    pd.DataFrame(
        history
    ).to_csv(
        D["logs"]
        / f"{safe_model_name(name)}_history_FINAL.csv",
        index=False
    )

    return model

trained = {}

load_report = []

for idx, (name, model) in enumerate(
    fresh_models().items()
):
    torch.manual_seed(
        SEED + idx
    )

    model = model.to(
        DEVICE
    )

    ckpt = checkpoint_path(
        name
    )

    if ckpt.exists():
        try:
            model.load_state_dict(
                load_state(
                    ckpt
                )
            )

            status = "loaded existing checkpoint"

        except Exception as e:
            print(
                f"Checkpoint load failed for {name}: {e}"
            )

            model = train_fallback(
                name,
                model
            )

            status = "fallback retrained from parquet"

    else:
        model = train_fallback(
            name,
            model
        )

        status = "checkpoint missing; retrained from parquet"

    trained[name] = model

    load_report.append({
        "Model": name,
        "Checkpoint": str(ckpt),
        "Status": status,
    })

load_report = pd.DataFrame(
    load_report
)

display(
    load_report
)

load_report.to_csv(
    D["tables"] / "model_checkpoint_audit_FINAL.csv",
    index=False
)

In [ ]:
# ============================================================
# 7. CLEAN FORECASTING RESULTS + EFFICIENCY
# ============================================================
result_tabs = []
prediction_store = {}
eff_rows = []

# Persistence
test_rows = [
    r for r in samples
    if r["split"] == "test"
]

y_true_test = np.vstack([
    r["y_raw"]
    for r in test_rows
])

persistence_pred = []

for r in test_rows:

    x_raw = x_scaler.inverse_transform(
        r["x"]
    )

    last_30s = (
        x_raw[
            -CFG["FORECAST_BLOCK_STEPS"]:,
            TRAFFIC_COLS.index("total")
        ]
        .sum()
    )

    persistence_pred.append(
        [last_30s] * NO
    )

persistence_pred = np.asarray(
    persistence_pred
)

result_tabs.append(
    metric_table(
        y_true_test,
        persistence_pred,
        "Persistence"
    )
)

prediction_store["Persistence"] = (
    y_true_test,
    persistence_pred
)

@torch.no_grad()
def efficiency_stats(
    model,
    loader,
    n_batches=30
):
    model.eval()

    batches = []

    for x, q, _, _, _ in loader:
        batches.append((
            x.to(DEVICE),
            q.to(DEVICE)
        ))

        if len(batches) >= n_batches:
            break

    for x, q in batches[:3]:
        _ = model(x, q)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    n_sample = 0

    for x, q in batches:
        _ = model(x, q)
        n_sample += len(x)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    dt = (
        time.perf_counter()
        - t0
    )

    return {
        "Params": sum(
            p.numel()
            for p in model.parameters()
        ),
        "Latency_ms_per_batch": (
            dt
            / max(len(batches), 1)
            * 1000
        ),
        "Latency_ms_per_sample": (
            dt
            / max(n_sample, 1)
            * 1000
        ),
    }

for name, model in trained.items():

    y_true, y_pred, _, _ = predict_loader(
        model,
        loaders["test"]
    )

    result_tabs.append(
        metric_table(
            y_true,
            y_pred,
            name
        )
    )

    prediction_store[name] = (
        y_true,
        y_pred
    )

    e = efficiency_stats(
        model,
        loaders["test"]
    )

    e["Model"] = name

    eff_rows.append(e)

results = pd.concat(
    result_tabs,
    ignore_index=True
)

efficiency_df = pd.DataFrame(
    eff_rows
)

results.to_csv(
    D["tables"] / "table_main_forecasting_results_FINAL.csv",
    index=False
)

efficiency_df.to_csv(
    D["tables"] / "table_model_efficiency_FINAL.csv",
    index=False
)

mean_results = (
    results
    .groupby(
        "Model",
        as_index=False
    )
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE_pct=("WAPE_%", "mean"),
        R2=("R2", "mean"),
    )
    .sort_values("MAE")
)

display(
    mean_results
)

In [ ]:
# ============================================================
# 8. ABLATION
# ============================================================
def get_or_train_gate_ablation():

    name = "A2_ReliabilityGate"

    model = VRATFormer(
        True,
        False
    ).to(DEVICE)

    ckpt = checkpoint_path(
        name
    )

    if ckpt.exists():
        try:
            model.load_state_dict(
                load_state(
                    ckpt
                )
            )

            print(
                "[cache] loaded",
                ckpt.name
            )

            return model

        except Exception as e:
            print(
                "A2 checkpoint load error:",
                e
            )

    return train_fallback(
        name,
        model
    )

a2_model = get_or_train_gate_ablation()

ablation_models = {
    "A0_Transformer":
        trained["Transformer"],

    "A1_ReliabilityConcat":
        trained[
            "Transformer+ReliabilityConcat"
        ],

    "A2_ReliabilityGate":
        a2_model,

    "A3_VRA-TFormer":
        trained["VRA-TFormer"],
}

ab_rows = []
ab_pred = {}

for name, model in ablation_models.items():

    yt, yp, _, _ = predict_loader(
        model,
        loaders["test"]
    )

    ab_rows.append(
        metric_table(
            yt,
            yp,
            name
        )
    )

    ab_pred[name] = (
        yt,
        yp
    )

ablation = pd.concat(
    ab_rows,
    ignore_index=True
)

ablation.to_csv(
    D["tables"] / "table_ablation_FINAL.csv",
    index=False
)

display(
    ablation.groupby(
        "Model",
        as_index=False
    )[
        ["MAE", "RMSE", "WAPE_%", "R2"]
    ].mean()
)

In [ ]:
# ============================================================
# 9. CLEAN WILCOXON + HOLM
# ============================================================
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

clean_mean_mae = (
    results
    .groupby("Model")["MAE"]
    .mean()
    .sort_values()
)

baseline_candidates = [
    m
    for m in clean_mean_mae.index
    if m not in {
        "VRA-TFormer",
        "Persistence"
    }
]

strongest_clean = baseline_candidates[0]

print(
    "Strongest clean comparator:",
    strongest_clean
)

yt_vra, yp_vra = prediction_store[
    "VRA-TFormer"
]

yt_base, yp_base = prediction_store[
    strongest_clean
]

clean_stat_rows = []
pvals = []

for j in range(NO):

    vra_err = np.abs(
        yt_vra[:, j]
        - yp_vra[:, j]
    )

    base_err = np.abs(
        yt_base[:, j]
        - yp_base[:, j]
    )

    try:
        stat, p = wilcoxon(
            vra_err,
            base_err,
            alternative="less"
        )
    except Exception:
        stat, p = np.nan, 1.0

    clean_stat_rows.append({
        "Horizon": f"{30*(j+1)}s",
        "Baseline": strongest_clean,
        "Baseline_MAE": base_err.mean(),
        "VRA_MAE": vra_err.mean(),
        "Improvement_%": (
            (base_err.mean() - vra_err.mean())
            / max(base_err.mean(), 1e-9)
            * 100
        ),
        "VRA_win_rate_%": (
            np.mean(
                vra_err < base_err
            )
            * 100
        ),
        "Wilcoxon_stat": stat,
        "p_raw": p,
    })

    pvals.append(p)

reject, p_holm, _, _ = multipletests(
    pvals,
    alpha=0.05,
    method="holm"
)

clean_stats = pd.DataFrame(
    clean_stat_rows
)

clean_stats["p_holm"] = p_holm
clean_stats["significant_0.05"] = reject

clean_stats.to_csv(
    D["tables"] / "table_clean_wilcoxon_holm_FINAL.csv",
    index=False
)

display(
    clean_stats
)

In [ ]:
# ============================================================
# 10. ROBUSTNESS — FINAL CORRECT PAIRED DESIGN
#
# degraded history -> model -> CLEAN future target
#
# Models used here must support variable sequence length:
# Transformer, reliability-concat Transformer, and VRA-TFormer.
# ============================================================

ROBUST_MODELS = {
    "Transformer":
        trained["Transformer"],

    "Transformer+ReliabilityConcat":
        trained[
            "Transformer+ReliabilityConcat"
        ],

    "VRA-TFormer":
        trained["VRA-TFormer"],
}

degraded_sequences = sorted(
    degraded_df["sequence"].unique()
)

FUTURE_STEPS = (
    CFG["FORECAST_BLOCK_STEPS"]
    * CFG["N_HORIZONS"]
)

# ------------------------------------------------------------
# Choose the largest history that supports enough EXISTING
# degradation sequences.
# ------------------------------------------------------------
seq_common_length = {}

for seq in degraded_sequences:

    clean_bins = set(
        clean_df.loc[
            clean_df["sequence"] == seq,
            "bin_idx"
        ].astype(int)
    )

    deg_lengths = []

    for (_, _), g in (
        degraded_df[
            degraded_df["sequence"] == seq
        ]
        .groupby(
            ["condition", "severity"]
        )
    ):
        deg_bins = set(
            g["bin_idx"].astype(int)
        )

        deg_lengths.append(
            len(
                clean_bins
                & deg_bins
            )
        )

    if deg_lengths:
        seq_common_length[seq] = min(
            deg_lengths
        )

print(
    "Common bins across all degradation scenarios:"
)

display(
    pd.DataFrame([
        {
            "sequence": k,
            "common_bins": v,
            "duration_min_approx": (
                v
                * CFG["BIN_SEC"]
                / 60
            )
        }
        for k, v in seq_common_length.items()
    ])
)

robust_lookback = None
robust_sequences = []

for candidate in CFG[
    "ROBUST_LOOKBACK_CANDIDATES"
]:

    eligible = [
        seq
        for seq, n in seq_common_length.items()
        if n >= (
            candidate
            + FUTURE_STEPS
            + 1
        )
    ]

    if (
        len(eligible)
        >= CFG[
            "MIN_ROBUST_SEQUENCES_DESIRED"
        ]
    ):
        robust_lookback = candidate
        robust_sequences = eligible
        break

if robust_lookback is None:

    for candidate in CFG[
        "ROBUST_LOOKBACK_CANDIDATES"
    ]:

        eligible = [
            seq
            for seq, n in seq_common_length.items()
            if n >= (
                candidate
                + FUTURE_STEPS
                + 1
            )
        ]

        if len(eligible) >= 1:
            robust_lookback = candidate
            robust_sequences = eligible
            break

if robust_lookback is None:
    raise RuntimeError(
        "Existing degraded clips are too short even for "
        "30 s history + 120 s future. "
        "No dataset is downloaded or rerun automatically."
    )

print("\n====================================")
print("Robustness lookback steps:", robust_lookback)
print(
    "Robustness history:",
    robust_lookback * CFG["BIN_SEC"],
    "seconds"
)
print("Sequences used:", robust_sequences)
print("====================================")

if robust_lookback < CFG["LOOKBACK_STEPS"]:
    print(
        "\nNOTE: The main forecasting experiment remains 5-min history.\n"
        "Only the controlled paired stress test uses a shorter history "
        "because the already-generated degradation clips are shorter.\n"
        "Transformer / Concat / VRA support variable history length."
    )

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def prepare_input_frame(df0):

    g = (
        df0
        .sort_values("bin_idx")
        .drop_duplicates("bin_idx")
        .copy()
    )

    for c in REL_COLS:
        g[c] = (
            g[c]
            .interpolate(
                limit_direction="both"
            )
            .fillna(
                rel_median[c]
            )
        )

    return g

def build_paired_rows(
    scenario_df,
    clean_reference,
    sequence,
    lookback
):
    deg = prepare_input_frame(
        scenario_df[
            scenario_df["sequence"]
            == sequence
        ]
    )

    ref = prepare_input_frame(
        clean_reference[
            clean_reference["sequence"]
            == sequence
        ]
    )

    common_bins = sorted(
        set(
            deg["bin_idx"].astype(int)
        )
        & set(
            ref["bin_idx"].astype(int)
        )
    )

    if not common_bins:
        return [], []

    deg = (
        deg[
            deg["bin_idx"]
            .isin(common_bins)
        ]
        .sort_values("bin_idx")
        .reset_index(drop=True)
    )

    ref = (
        ref[
            ref["bin_idx"]
            .isin(common_bins)
        ]
        .sort_values("bin_idx")
        .reset_index(drop=True)
    )

    n = min(
        len(deg),
        len(ref)
    )

    deg = deg.iloc[:n].copy()
    ref = ref.iloc[:n].copy()

    X_deg = x_scaler.transform(
        deg[TRAFFIC_COLS]
    ).astype(np.float32)

    Q_deg = q_scaler.transform(
        deg[REL_COLS]
    ).astype(np.float32)

    X_clean = x_scaler.transform(
        ref[TRAFFIC_COLS]
    ).astype(np.float32)

    Q_clean = q_scaler.transform(
        ref[REL_COLS]
    ).astype(np.float32)

    clean_total = (
        ref["total"]
        .values
        .astype(float)
    )

    degraded_rows = []
    matched_clean_rows = []

    for t in range(
        lookback,
        n - FUTURE_STEPS + 1
    ):

        y_raw = np.array([
            clean_total[
                t + h*B :
                t + (h+1)*B
            ].sum()
            for h in range(H)
        ], dtype=np.float32)

        y_scaled = (
            y_scaler
            .transform(
                y_raw.reshape(-1, 1)
            )
            .reshape(-1)
            .astype(np.float32)
        )

        base = {
            "sequence": sequence,
            "t": int(t),
            "y_raw": y_raw,
            "y": y_scaled,
        }

        degraded_rows.append({
            **base,
            "x": X_deg[
                t-lookback:t
            ],
            "q": Q_deg[
                t-lookback:t
            ],
        })

        matched_clean_rows.append({
            **base,
            "x": X_clean[
                t-lookback:t
            ],
            "q": Q_clean[
                t-lookback:t
            ],
        })

    return (
        degraded_rows,
        matched_clean_rows
    )

class RobustDS(Dataset):

    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(
            self.rows
        )

    def __getitem__(self, i):
        r = self.rows[i]

        return (
            torch.tensor(
                r["x"],
                dtype=torch.float32
            ),
            torch.tensor(
                r["q"],
                dtype=torch.float32
            ),
            torch.tensor(
                r["y"],
                dtype=torch.float32
            ),
            r["sequence"],
            int(r["t"]),
        )

def loader_from_rows(rows):
    return DataLoader(
        RobustDS(rows),
        batch_size=CFG["BATCH_SIZE"],
        shuffle=False,
        num_workers=0
    )

# ------------------------------------------------------------
# Evaluate each scenario against its EXACT matched clean windows.
# ------------------------------------------------------------
robust_full_rows = []
robust_summary_rows = []
robust_pred_store = {}

scenario_groups = list(
    degraded_df
    .groupby(
        ["condition", "severity"]
    )
)

for (
    condition,
    severity
), scenario_df in tqdm(
    scenario_groups,
    desc="Paired robustness scenarios"
):

    degraded_rows = []
    clean_rows = []

    for seq in robust_sequences:

        dr, cr = build_paired_rows(
            scenario_df,
            clean_df,
            seq,
            robust_lookback
        )

        degraded_rows.extend(
            dr
        )

        clean_rows.extend(
            cr
        )

    if len(degraded_rows) == 0:
        print(
            "SKIP",
            condition,
            severity,
            "no valid paired windows"
        )
        continue

    print(
        f"{condition:12s}",
        f"severity={severity}",
        f"paired windows={len(degraded_rows)}"
    )

    deg_loader = loader_from_rows(
        degraded_rows
    )

    clean_loader = loader_from_rows(
        clean_rows
    )

    for model_name, model in ROBUST_MODELS.items():

        y_clean, p_clean, seq_c, t_c = predict_loader(
            model,
            clean_loader
        )

        y_deg, p_deg, seq_d, t_d = predict_loader(
            model,
            deg_loader
        )

        # Identical targets are expected.
        if not np.allclose(
            y_clean,
            y_deg
        ):
            raise RuntimeError(
                "Paired robustness target mismatch."
            )

        clean_tab = metric_table(
            y_clean,
            p_clean,
            model_name
        )

        deg_tab = metric_table(
            y_deg,
            p_deg,
            model_name
        )

        for j in range(NO):

            clean_mae_h = clean_tab.iloc[j]["MAE"]
            deg_mae_h = deg_tab.iloc[j]["MAE"]

            robust_full_rows.append({
                "Model": model_name,
                "Condition": condition,
                "Severity": int(severity),
                "Horizon": f"{30*(j+1)}s",
                "Clean_MAE": clean_mae_h,
                "Degraded_MAE": deg_mae_h,
                "Clean_RMSE": clean_tab.iloc[j]["RMSE"],
                "Degraded_RMSE": deg_tab.iloc[j]["RMSE"],
                "Clean_WAPE_%": clean_tab.iloc[j]["WAPE_%"],
                "Degraded_WAPE_%": deg_tab.iloc[j]["WAPE_%"],
                "Relative_Degradation_%": (
                    (
                        deg_mae_h
                        - clean_mae_h
                    )
                    / max(
                        clean_mae_h,
                        1e-9
                    )
                    * 100
                ),
                "N_windows": len(
                    degraded_rows
                ),
                "Robust_Lookback_steps": robust_lookback,
                "Robust_Lookback_sec": (
                    robust_lookback
                    * CFG["BIN_SEC"]
                ),
            })

        clean_mae_all = np.mean(
            np.abs(
                y_clean
                - p_clean
            )
        )

        deg_mae_all = np.mean(
            np.abs(
                y_deg
                - p_deg
            )
        )

        robust_summary_rows.append({
            "Model": model_name,
            "Condition": condition,
            "Severity": int(severity),
            "Clean_MAE": clean_mae_all,
            "Degraded_MAE": deg_mae_all,
            "Relative_Degradation_%": (
                (
                    deg_mae_all
                    - clean_mae_all
                )
                / max(
                    clean_mae_all,
                    1e-9
                )
                * 100
            ),
            "N_windows": len(
                degraded_rows
            ),
            "N_sequences": len(
                robust_sequences
            ),
            "Robust_Lookback_steps": robust_lookback,
            "Robust_Lookback_sec": (
                robust_lookback
                * CFG["BIN_SEC"]
            ),
        })

        # Store per-window error averaged over horizons for significance.
        robust_pred_store[
            (
                condition,
                int(severity),
                model_name
            )
        ] = {
            "y": y_deg,
            "pred": p_deg,
            "window_error": (
                np.abs(
                    y_deg
                    - p_deg
                )
                .mean(axis=1)
            ),
        }

robustness_full = pd.DataFrame(
    robust_full_rows
)

robust_summary = pd.DataFrame(
    robust_summary_rows
)

if len(robust_summary) == 0:
    raise RuntimeError(
        "No paired robustness results were produced."
    )

robustness_full.to_csv(
    D["tables"] / "table_robustness_full_FINAL.csv",
    index=False
)

robust_summary.to_csv(
    D["tables"] / "table_robustness_summary_FINAL.csv",
    index=False
)

print("\n✅ ROBUSTNESS COMPLETE")
display(
    robust_summary.sort_values(
        [
            "Condition",
            "Severity",
            "Degraded_MAE"
        ]
    )
)

In [ ]:
# ============================================================
# 11. ROBUSTNESS STATISTICAL TESTS
# VRA vs Transformer and VRA vs ReliabilityConcat
# per degradation scenario, window-level mean absolute error.
# ============================================================
robust_stat_rows = []

comparators = [
    "Transformer",
    "Transformer+ReliabilityConcat",
]

for (
    condition,
    severity
) in sorted(
    set(
        (k[0], k[1])
        for k in robust_pred_store.keys()
    )
):

    vra_key = (
        condition,
        severity,
        "VRA-TFormer"
    )

    if vra_key not in robust_pred_store:
        continue

    vra_err = robust_pred_store[
        vra_key
    ]["window_error"]

    for comparator in comparators:

        base_key = (
            condition,
            severity,
            comparator
        )

        if base_key not in robust_pred_store:
            continue

        base_err = robust_pred_store[
            base_key
        ]["window_error"]

        n = min(
            len(vra_err),
            len(base_err)
        )

        ve = vra_err[:n]
        be = base_err[:n]

        try:
            stat, p = wilcoxon(
                ve,
                be,
                alternative="less"
            )
        except Exception:
            stat, p = np.nan, 1.0

        robust_stat_rows.append({
            "Condition": condition,
            "Severity": severity,
            "Comparator": comparator,
            "N_windows": n,
            "Comparator_MAE": be.mean(),
            "VRA_MAE": ve.mean(),
            "VRA_Improvement_%": (
                (
                    be.mean()
                    - ve.mean()
                )
                / max(
                    be.mean(),
                    1e-9
                )
                * 100
            ),
            "VRA_win_rate_%": (
                np.mean(
                    ve < be
                )
                * 100
            ),
            "Wilcoxon_stat": stat,
            "p_raw": p,
        })

robust_stats = pd.DataFrame(
    robust_stat_rows
)

if len(robust_stats):

    reject, p_adj, _, _ = multipletests(
        robust_stats["p_raw"].values,
        alpha=0.05,
        method="holm"
    )

    robust_stats["p_holm"] = p_adj
    robust_stats["significant_0.05"] = reject

robust_stats.to_csv(
    D["tables"] / "table_robustness_wilcoxon_holm_FINAL.csv",
    index=False
)

display(
    robust_stats
)

In [ ]:
# ============================================================
# 12. LEARNED RELIABILITY + STORY DIAGNOSTIC
# ============================================================
vra_model = trained[
    "VRA-TFormer"
]

vra_model.eval()

learned_rows = []

with torch.no_grad():

    for x, q, y, seq, t in loaders["test"]:

        pred0, rel0, lam0 = vra_model(
            x.to(DEVICE),
            q.to(DEVICE),
            return_reliability=True
        )

        rel0 = (
            rel0
            .squeeze(-1)
            .cpu()
            .numpy()
        )

        for i in range(
            len(seq)
        ):
            learned_rows.append({
                "sequence": seq[i],
                "t": int(t[i]),
                "mean_learned_reliability":
                    float(rel0[i].mean()),
                "min_learned_reliability":
                    float(rel0[i].min()),
                "max_learned_reliability":
                    float(rel0[i].max()),
                "lambda_attention":
                    float(
                        lam0.detach().cpu()
                    ),
            })

learned_rel = pd.DataFrame(
    learned_rows
)

learned_rel.to_csv(
    D["tables"] / "learned_reliability_FINAL.csv",
    index=False
)

avg_rd = (
    robust_summary
    .groupby(
        "Model",
        as_index=False
    )["Relative_Degradation_%"]
    .mean()
    .sort_values(
        "Relative_Degradation_%"
    )
)

avg_deg_mae = (
    robust_summary
    .groupby(
        "Model",
        as_index=False
    )["Degraded_MAE"]
    .mean()
    .sort_values(
        "Degraded_MAE"
    )
)

display(avg_rd)
display(avg_deg_mae)

best_clean_model = (
    mean_results.iloc[0]["Model"]
)

best_robust_rd_model = (
    avg_rd.iloc[0]["Model"]
)

best_degraded_mae_model = (
    avg_deg_mae.iloc[0]["Model"]
)

print("\n====================================")
print("EMPIRICAL STORY CHECK")
print("Best clean MAE:", best_clean_model)
print(
    "Lowest average relative degradation:",
    best_robust_rd_model
)
print(
    "Lowest average degraded MAE:",
    best_degraded_mae_model
)
print("====================================")

if (
    best_robust_rd_model == "VRA-TFormer"
    or best_degraded_mae_model == "VRA-TFormer"
):
    print(
        "✅ Robustness-oriented VRA story is empirically supportable, "
        "subject to significance results."
    )
else:
    print(
        "⚠️ Do NOT force a VRA superiority claim. "
        "Consider pivoting the paper to visual-reliability augmentation "
        "or a controlled comparative study."
    )

In [ ]:
# ============================================================
# 13. PUBLICATION FIGURES
# ============================================================
def savefig(fig, name):
    fig.savefig(
        D["figures"] / f"{name}.png",
        dpi=CFG["FIG_DPI"],
        bbox_inches="tight"
    )
    fig.savefig(
        D["figures"] / f"{name}.pdf",
        bbox_inches="tight"
    )

# Fig 4 FINAL — clean MAE by horizon
fig, ax = plt.subplots(
    figsize=(7.4, 4.1)
)

for name in [
    "Persistence",
    "LSTM",
    "GRU",
    "Transformer",
    "PatchTST",
    "iTransformer",
    "Transformer+ReliabilityConcat",
    "VRA-TFormer",
]:
    z = results[
        results["Model"] == name
    ]

    if len(z):
        ax.plot(
            z["Horizon"],
            z["MAE"],
            marker="o",
            linewidth=1.1,
            label=name
        )

ax.set_xlabel(
    "Forecast horizon"
)
ax.set_ylabel(
    "MAE (vehicles / 30-s block)"
)
ax.set_title(
    "Clean-condition multi-horizon forecasting"
)
ax.legend(
    ncol=2,
    frameon=False
)
fig.tight_layout()
savefig(
    fig,
    "Fig4_FINAL_clean_forecasting_MAE"
)
plt.show()

# Fig 5 FINAL — ablation
abl_mean = (
    ablation
    .groupby(
        "Model",
        as_index=False
    )["MAE"]
    .mean()
)

fig, ax = plt.subplots(
    figsize=(6.4, 3.9)
)
ax.bar(
    abl_mean["Model"],
    abl_mean["MAE"]
)
ax.set_ylabel(
    "Mean MAE"
)
ax.set_title(
    "Ablation of visual-reliability mechanisms"
)
ax.tick_params(
    axis="x",
    rotation=25
)
fig.tight_layout()
savefig(
    fig,
    "Fig5_FINAL_ablation"
)
plt.show()

# Fig 6 FINAL — relative degradation by condition/severity.
plot_rd = robust_summary.copy()

# Combine condition + severity on x-axis.
plot_rd["Scenario"] = (
    plot_rd["Condition"].astype(str)
    + "-S"
    + plot_rd["Severity"].astype(str)
)

scenario_order = (
    plot_rd[
        ["Condition", "Severity", "Scenario"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Condition", "Severity"]
    )["Scenario"]
    .tolist()
)

models_order = [
    "Transformer",
    "Transformer+ReliabilityConcat",
    "VRA-TFormer",
]

x = np.arange(
    len(scenario_order)
)
width = 0.25

fig, ax = plt.subplots(
    figsize=(9.0, 4.4)
)

for i, model_name in enumerate(
    models_order
):
    d0 = (
        plot_rd[
            plot_rd["Model"] == model_name
        ]
        .set_index("Scenario")
        .reindex(
            scenario_order
        )
    )

    ax.bar(
        x + (
            i
            - (len(models_order)-1)/2
        ) * width,
        d0["Relative_Degradation_%"],
        width=width,
        label=model_name
    )

ax.axhline(
    0,
    linewidth=0.8
)
ax.set_xticks(x)
ax.set_xticklabels(
    scenario_order,
    rotation=25
)
ax.set_ylabel(
    "Relative MAE degradation (%)"
)
ax.set_title(
    "Paired robustness under visual degradation"
)
ax.legend(
    frameon=False
)
fig.tight_layout()
savefig(
    fig,
    "Fig6_FINAL_robustness_relative_degradation"
)
plt.show()

# Fig 7 FINAL — average clean vs degraded MAE by robustness model.
rob_plot = (
    robust_summary
    .groupby(
        "Model",
        as_index=False
    )
    .agg(
        Clean_MAE=("Clean_MAE", "mean"),
        Degraded_MAE=("Degraded_MAE", "mean"),
    )
)

fig, ax = plt.subplots(
    figsize=(6.4, 4.0)
)

xx = np.arange(
    len(rob_plot)
)
ww = 0.35

ax.bar(
    xx - ww/2,
    rob_plot["Clean_MAE"],
    width=ww,
    label="Matched clean"
)

ax.bar(
    xx + ww/2,
    rob_plot["Degraded_MAE"],
    width=ww,
    label="Degraded input"
)

ax.set_xticks(xx)
ax.set_xticklabels(
    rob_plot["Model"],
    rotation=20
)
ax.set_ylabel(
    "Mean absolute error"
)
ax.set_title(
    "Matched clean vs degraded forecasting error"
)
ax.legend(
    frameon=False
)
fig.tight_layout()
savefig(
    fig,
    "Fig7_FINAL_matched_clean_vs_degraded"
)
plt.show()

# Fig 8 FINAL — representative clean 30-s forecast.
y0, p0 = prediction_store[
    "VRA-TFormer"
]

_, pb0 = prediction_store[
    strongest_clean
]

nshow = min(
    120,
    len(y0)
)

fig, ax = plt.subplots(
    figsize=(8.0, 3.8)
)

ax.plot(
    np.arange(nshow),
    y0[:nshow, 0],
    label="Ground truth",
    linewidth=1.2
)

ax.plot(
    np.arange(nshow),
    pb0[:nshow, 0],
    label=strongest_clean,
    linestyle="--",
    linewidth=1.0
)

ax.plot(
    np.arange(nshow),
    p0[:nshow, 0],
    label="VRA-TFormer",
    linewidth=1.1
)

ax.set_xlabel(
    "Test window"
)
ax.set_ylabel(
    "Vehicles / next 30 s"
)
ax.set_title(
    "Representative 30-s-ahead clean forecast"
)
ax.legend(
    frameon=False
)
fig.tight_layout()
savefig(
    fig,
    "Fig8_FINAL_clean_forecast_trace"
)
plt.show()

print(
    "Publication figures saved to:",
    D["figures"]
)

In [ ]:
# ============================================================
# 14. FINAL EXCEL + DIGEST + ZIP
# ============================================================
# Dataset summary rebuilt from clean cache.
dataset_rows = []

for seq, g in clean_df.groupby(
    "sequence"
):
    dataset_rows.append({
        "sequence": seq,
        "bins": len(g),
        "duration_min": (
            g["duration_s"].iloc[0] / 60
            if "duration_s" in g.columns
            else len(g) * CFG["BIN_SEC"] / 60
        ),
        "vehicles_counted": g["total"].sum(),
        "car": g["car"].sum(),
        "motorcycle": g["motorcycle"].sum(),
        "bus": g["bus"].sum(),
        "truck": g["truck"].sum(),
        "mean_conf": g["mean_conf"].mean(),
        "frame_availability":
            g["frame_availability"].mean(),
    })

dataset_summary = pd.DataFrame(
    dataset_rows
)

dataset_summary.to_csv(
    D["tables"] / "table_dataset_summary_FINAL.csv",
    index=False
)

xlsx_path = (
    D["tables"]
    / "SmartTraffic_IJIES_All_Tables_FINAL.xlsx"
)

with pd.ExcelWriter(
    xlsx_path,
    engine="openpyxl"
) as writer:

    audit.to_excel(
        writer,
        sheet_name="Asset_Audit",
        index=False
    )

    dataset_summary.to_excel(
        writer,
        sheet_name="Dataset_Summary",
        index=False
    )

    split_summary.to_excel(
        writer,
        sheet_name="Forecast_Splits",
        index=False
    )

    results.to_excel(
        writer,
        sheet_name="Main_Results",
        index=False
    )

    mean_results.to_excel(
        writer,
        sheet_name="Mean_Results",
        index=False
    )

    efficiency_df.to_excel(
        writer,
        sheet_name="Efficiency",
        index=False
    )

    ablation.to_excel(
        writer,
        sheet_name="Ablation",
        index=False
    )

    clean_stats.to_excel(
        writer,
        sheet_name="Clean_Wilcoxon",
        index=False
    )

    scenario_inventory.to_excel(
        writer,
        sheet_name="Degradation_Inventory",
        index=False
    )

    robustness_full.to_excel(
        writer,
        sheet_name="Robustness_Full",
        index=False
    )

    robust_summary.to_excel(
        writer,
        sheet_name="Robustness_Summary",
        index=False
    )

    robust_stats.to_excel(
        writer,
        sheet_name="Robustness_Wilcoxon",
        index=False
    )

    learned_rel.to_excel(
        writer,
        sheet_name="Learned_Reliability",
        index=False
    )

    load_report.to_excel(
        writer,
        sheet_name="Checkpoint_Audit",
        index=False
    )

    avg_rd.to_excel(
        writer,
        sheet_name="Average_Robustness",
        index=False
    )

# ------------------------------------------------------------
# Factual digest
# ------------------------------------------------------------
vra_clean = (
    mean_results[
        mean_results["Model"]
        == "VRA-TFormer"
    ]
    .iloc[0]
)

avg_rd_sorted = (
    avg_rd
    .sort_values(
        "Relative_Degradation_%"
    )
)

robust_best = (
    avg_rd_sorted.iloc[0]
)

digest_parts = [
    "# SmartTraffic IJIES — FINAL Resume Result Digest",
    "",
    "## Reproducibility / cache audit",
    audit.to_markdown(index=False),
    "",
    "## Forecast split",
    split_summary.to_markdown(index=False),
    "",
    "## Mean clean forecasting results",
    mean_results.to_markdown(index=False),
    "",
    "## Clean statistical comparison",
    clean_stats.to_markdown(index=False),
    "",
    "## Ablation",
    (
        ablation
        .groupby(
            "Model",
            as_index=False
        )[
            [
                "MAE",
                "RMSE",
                "WAPE_%",
                "R2",
            ]
        ]
        .mean()
        .to_markdown(index=False)
    ),
    "",
    "## Robustness configuration",
    (
        f"- Robustness history: "
        f"**{robust_lookback} steps "
        f"({robust_lookback * CFG['BIN_SEC']} s)**"
    ),
    (
        f"- Robustness sequences: "
        f"**{', '.join(robust_sequences)}**"
    ),
    (
        f"- Number of degradation scenarios: "
        f"**{robust_summary[['Condition','Severity']].drop_duplicates().shape[0]}**"
    ),
    "",
    "## Robustness summary",
    robust_summary.to_markdown(index=False),
    "",
    "## Average relative degradation",
    avg_rd_sorted.to_markdown(index=False),
    "",
    "## Robustness significance",
    robust_stats.to_markdown(index=False),
    "",
    "## Efficiency",
    efficiency_df.to_markdown(index=False),
    "",
    "## Factual manuscript notes",
    (
        f"- Best mean clean-MAE model: "
        f"**{mean_results.iloc[0]['Model']}** "
        f"(MAE={mean_results.iloc[0]['MAE']:.4f})."
    ),
    (
        f"- VRA-TFormer mean clean MAE: "
        f"**{vra_clean['MAE']:.4f}**."
    ),
    (
        f"- VRA-TFormer mean clean WAPE: "
        f"**{vra_clean['WAPE_pct']:.2f}%**."
    ),
    (
        f"- Lowest mean relative degradation: "
        f"**{robust_best['Model']}** "
        f"({robust_best['Relative_Degradation_%']:.2f}%)."
    ),
    (
        f"- Lowest average degraded-MAE model: "
        f"**{best_degraded_mae_model}**."
    ),
    (
        f"- Learned attention reliability strength lambda: "
        f"**{learned_rel['lambda_attention'].iloc[0]:.4f}**."
    ),
]

if (
    best_robust_rd_model == "VRA-TFormer"
    or best_degraded_mae_model == "VRA-TFormer"
):
    digest_parts += [
        "",
        "## Suggested empirical framing",
        (
            "The robustness-oriented VRA-TFormer story is supported "
            "by at least one aggregate robustness criterion. "
            "Do not claim universal clean-condition superiority."
        ),
    ]
else:
    digest_parts += [
        "",
        "## Suggested empirical framing",
        (
            "Do not claim VRA-TFormer superiority. "
            "The safer manuscript framing is visual-reliability "
            "augmentation / controlled comparison, with the empirically "
            "best clean and robustness variants reported honestly."
        ),
    ]

digest_path = (
    ROOT
    / "SmartTraffic_IJIES_Result_Digest_FINAL.md"
)

digest_path.write_text(
    "\n".join(
        digest_parts
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# Manifest + ZIP
# ------------------------------------------------------------
manifest = {
    "project_root": str(ROOT),
    "final_excel": str(xlsx_path),
    "final_digest": str(digest_path),
    "robust_lookback_steps": int(robust_lookback),
    "robust_sequences": robust_sequences,
    "best_clean_model": str(best_clean_model),
    "best_robust_rd_model": str(best_robust_rd_model),
    "best_degraded_mae_model": str(best_degraded_mae_model),
    "figures": [
        str(p)
        for p in sorted(
            D["figures"].glob(
                "*FINAL*"
            )
        )
    ],
    "tables": [
        str(p)
        for p in sorted(
            D["tables"].glob(
                "*FINAL*"
            )
        )
    ],
}

manifest_path = (
    ROOT
    / "artifact_manifest_FINAL.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8"
)

zip_path = (
    ROOT
    / "SmartTraffic_IJIES_Paper_Artifacts_FINAL.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for folder in [
        D["figures"],
        D["tables"],
        D["logs"],
    ]:
        for p in folder.rglob("*"):
            if (
                p.is_file()
                and (
                    "FINAL" in p.name
                    or p.suffix.lower()
                    in {".csv", ".json"}
                )
            ):
                zf.write(
                    p,
                    arcname=str(
                        p.relative_to(ROOT)
                    )
                )

    zf.write(
        digest_path,
        arcname=digest_path.name
    )

    zf.write(
        manifest_path,
        arcname=manifest_path.name
    )

print("\n============================================")
print("FINAL PIPELINE COMPLETE")
print("============================================")
print("Excel :", xlsx_path)
print("Digest:", digest_path)
print("ZIP   :", zip_path)
print("Figures:", D["figures"])
print("\nUPLOAD THESE 3 FILES BACK TO CHAT:")
print("1.", xlsx_path.name)
print("2.", digest_path.name)
print("3.", zip_path.name)